# 📒 MODULE 30 — VISION TRANSFORMERS

## 30.2 — Image → Patches

🔥🔥🔥 **MUST KNOW**

The central idea of ViT is to transform a 2D image into a **sequence of visual tokens** that a Transformer can process. The original ViT formulation uses fixed-size, non-overlapping image patches, linearly embeds them, adds a `[CLS]` token and positional embeddings, and feeds the resulting sequence into a Transformer encoder. ([arXiv][1])

---

# 1. 🧠 The Complete Pipeline

```text
IMAGE
  ↓
Split into patches
  ↓
Patch embeddings
  ↓
Add [CLS] token
  ↓
Add positional embeddings
  ↓
Transformer Encoder
  ↓
[CLS] representation
  ↓
Classification Head
  ↓
Class prediction
```

![Image](https://images.openai.com/static-rsc-4/az3kbAC8KVv3hFtHpRczePHLh2-GbtYgTzEeTlAZ-QKiCeivi8K6zwniXJpiFfBMwR8doWRxZzR4J0J1OQO_nXgvREORqvXyEAMmMlJ28pEBYL8BPXPRYGcu_jDrECyLKO3A09CGbilO0FEdvjYCuAk5ofJLwL1wR_dbzbeJ81ksuZuO7oKwXEG0jesIOAJF?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/ap3KId2CKXsCv4qmb9mtsbR83KRIlG_dBBAJmA-_wmBbpvo7R19pCM3gysrgQEGWpPaWHuTRcPNIscz7nu5vDn6neP1WcRW8SgJQyY2F3b8Clc_6bpJEoH745_5qNDDUoyjzeYhnjlQkeZU9T44p39Q2Y8cJmYnavtiUJrXl8aL1dUOCP0HUrIfdDuf-IQOg?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/XjLGV7IWc5NviPJevqI-liGt7g05y7T4sBAYRvdQ_bdTLn4Q5BMWXKgMAmQbCnl3aFHeoqEwhAO2bKPdPgxC3T-vwC-j2RBvzJyn9kF_Kqzwmt_H04hIbZIrzERaCodm17l67vyk-q04PcEvKJxtljfm7nFUamBN0kbg1MlbnR3ytQIXQTZ60uq5kf_cJzVN?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/j2MdKRClEV2cwr-3jSAMYsXp1oXLGS0JtRcYmQgBiGr1XzvhtQqqAWKOAT9_1nxwgu83s164LIHSqguQDrPqfiI8RyG7VTShmIuNN5PfprpIYI3QJz5v7vHXQu2fsRmBusRQJn4oNPznlDm5AuugydSP0MZfEjFQM7h7aNik1_pM4aqVmNOKh761yINFmlMt?purpose=fullsize)

The original ViT paper is the primary reference for this pipeline. ([arXiv][1])

---

# 2. 🖼️ Image → Patches

A normal image is represented as a spatial grid:

```text
Image
┌────────────────────────┐
│                        │
│       Pixels           │
│                        │
│                        │
└────────────────────────┘
```

ViT divides this image into **fixed-size, non-overlapping patches**.

For example:

```text
┌────┬────┬────┬────┐
│ P1 │ P2 │ P3 │ P4 │
├────┼────┼────┼────┤
│ P5 │ P6 │ P7 │ P8 │
├────┼────┼────┼────┤
│ P9 │P10 │P11 │P12 │
├────┼────┼────┼────┤
│P13 │P14 │P15 │P16 │
└────┴────┴────┴────┘
```

The patches become the basic visual units processed by the Transformer.

Hugging Face describes this as analogous to tokenizing text into words/tokens. ([Hugging Face][2])

---

# 3. 🔢 Number of Patches

For an image:

$$
H\times W
$$

and square patch size:

$$
P\times P
$$

the number of patches is:

$$
\boxed{N=\frac{H}{P}\times\frac{W}{P}}
$$

or:

$$
\boxed{N=\frac{HW}{P^2}}
$$

### Example

Image:

$$
224\times224
$$

Patch:

$$
16\times16
$$

Therefore:

$$
\frac{224}{16}=14
$$

patches along each dimension.

Hence:

$$
14\times14=\boxed{196}
$$

So:

> **224×224 image + 16×16 patches = 196 patches.**

Hugging Face's ViT documentation gives the same 224×224 → 196 patches example. ([Hugging Face][2])

---

# 4. 🧩 What Is Inside One Patch?

For an RGB image:

```text
Channels = 3
```

A \(16\times16\) patch contains:

$$
16\times16\times3
$$

$$
=\boxed{768}
$$

pixel values.

So:

```text
16 × 16 × 3
      ↓
768 values
```

Conceptually:

```text
RGB Patch
┌─────────────────┐
│ R G B R G B ... │
│ R G B R G B ... │
│       ...       │
└─────────────────┘
        ↓
  768 values
```

---

# 5. 🔄 Patch → Patch Embedding

The raw pixels are not directly treated as the final Transformer representation.

The patch is transformed into an **embedding vector**.

```text
Image Patch
     ↓
Flatten / projection
     ↓
Patch Embedding
     ↓
Vector
```

Conceptually:

$$
\mathbf{z}=\mathbf{x}\mathbf{E}+\mathbf{b}
$$

where:

* \(\mathbf{x}\) = flattened patch
* \(\mathbf{E}\) = learnable projection
* \(\mathbf{b}\) = bias, depending on implementation
* \(\mathbf{z}\) = patch embedding

The original ViT describes this as a **linear projection of flattened image patches**. ([arXiv][1])

### Mental model

> **Raw patch → learned visual representation**

---

# 6. 📐 Numerical Example

Suppose:

```text
Image       = 224 × 224 × 3
Patch       = 16 × 16
Embedding   = 768
```

Then:

### Number of patches

$$
N=196
$$

### Raw values per patch

$$
16\times16\times3=768
$$

After projection:

```text
196 patches
     ↓
196 embedding vectors
     ↓
each = 768 dimensions
```

So the patch sequence has shape:

$$
\boxed{196\times768}
$$

For a batch of \(B\) images:

$$
\boxed{B\times196\times768}
$$

---

# 7. 🧠 Patch Embedding ≈ Token Embedding

This analogy is extremely useful.

### NLP

```text
Sentence
 ↓
Words / subwords
 ↓
Tokens
 ↓
Token embeddings
 ↓
Transformer
```

### ViT

```text
Image
 ↓
Patches
 ↓
Visual tokens
 ↓
Patch embeddings
 ↓
Transformer
```

So:

| NLP             | ViT             |
| --------------- | --------------- |
| Word/subword    | Image patch     |
| Token           | Visual token    |
| Token embedding | Patch embedding |
| Transformer     | Transformer     |

This is the conceptual bridge between **language Transformers and Vision Transformers**.

---

# 8. 📍 Why Positional Embeddings?

Now suppose we have:

```text
[P1] [P2] [P3] [P4]
```

The Transformer needs to know where these patches originated spatially.

Consider:

```text
┌────┬────┐
│ P1 │ P2 │
├────┼────┤
│ P3 │ P4 │
└────┴────┘
```

versus:

```text
┌────┬────┐
│ P3 │ P1 │
├────┼────┤
│ P4 │ P2 │
└────┴────┘
```

The same patches are present, but their **spatial arrangement is different**.

Therefore, positional information is added.

```text
Patch Embedding
      +
Position Embedding
      ↓
Transformer Input
```

Hugging Face's ViT documentation notes that the model adds **learnable absolute position embeddings** because the Transformer does not inherently know the ordering/location of the patches. ([Hugging Face][2])

---

# 9. 🧠 Patch Embedding vs Positional Embedding

🔥 **Very important distinction**

| Embedding                | Main question                                          |
| ------------------------ | ------------------------------------------------------ |
| **Patch embedding**      | **What** visual information does this patch represent? |
| **Positional embedding** | **Where** is this patch located?                       |

Mental model:

```text
              PATCH
                │
       ┌────────┴────────┐
       ↓                 ↓
 Visual information   Position
       ↓                 ↓
 Patch embedding    Positional embedding
       └────────┬────────┘
                ↓
        Transformer input
```

---

# 10. 🟦 The `[CLS]` Token

The simplified pipeline often says:

```text
Image
 ↓
Patches
 ↓
Patch embeddings
 ↓
Positional embeddings
 ↓
Transformer
```

But the original ViT classification architecture includes an additional learnable **`[CLS]` token**.

The sequence becomes:

```text
[CLS] [P1] [P2] [P3] ... [P196]
```

Therefore:

$$
196+1=\boxed{197}
$$

tokens.

The final representation corresponding to `[CLS]` is used as the whole-image representation for classification. ([Hugging Face][2])

![Image](https://images.openai.com/static-rsc-4/iG1_t9lsamaFakSCQc1ocMpTK5dsgwwB5lDFNc_PD16Ys27hpqYZiTPeyv3eAMPy5fvOfux7Vm8yXs5Opnn4B-BCYMJyUhNQjrMImze10giuZjs9nmmkWdADiHcjZdiyhg941MwDeZEmCoU24yj5DHL-NSzcBKzly5ruDifpZywTQntWeVawWp0t3m8R-Fcp?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/NwEbGnjdvtme4n878q1HrUEU7BLkb7CFcbXdwc36b8qam7WfKtmZTmt4V4K1svKIJeVS_EdBNiRj4E-7sog920v481rHDHzXghodYFQXaMj-Mm5-VRw9iOk_ol2yyeXMJyaVqqc2UaMrbNgdNHDRoefAScuJ5dbmimVSWkKuytrtcFWymetnfULxloulF3eF?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/n3Vnn0MzoucXGZDgNiQu_jNqkE_97AW0frWcEQEh5vyTpKJb9EWovfWOVKN1yweR1-V-RIP-HoYCdpANG8ChNanF9zKgPs6U0misTrSv9XQJPpd-tkjMW2Wz6JhPaGw4tMSjgV0ex-cHkGgUVNiZz6uxukbZ60bVrUWWZyvBRPVGYGX-IwGXkUIFxcsOXEhY?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/LKu7lylRNwdgzaIE7Og4E6b2uSod3FH0aLy01m1E7iISaBRdByLNPrxAu4eD5BFGrgLecerdOLUcYmcb--UGhV1lFxVAEuanAaMDMWg3JHVgUdMhU1TkbPBRLdkgFq_lFMBOX3-tBa8TQV4aVohvtkRHZ5JtHgELunyean-Q9_KF8nlAyWTnfoTlSY0Y9JuF?purpose=fullsize)

---

# 11. 📐 Complete Shape Flow

For a typical ViT-Base-style configuration:

```text
Input image
[B, 3, 224, 224]
        ↓
Split into 16×16 patches
        ↓
196 patches
        ↓
Patch projection
        ↓
[B, 196, 768]
        ↓
Add [CLS]
        ↓
[B, 197, 768]
        ↓
Add positional embeddings
        ↓
[B, 197, 768]
        ↓
Transformer Encoder
        ↓
[B, 197, 768]
        ↓
Take [CLS]
        ↓
[B, 768]
        ↓
Classification Head
        ↓
[B, num_classes]
```

The Hugging Face configuration for `google/vit-base-patch16-224` uses image size 224, patch size 16, 3 channels, and hidden size 768. ([Hugging Face][3])

---

# 12. 🧮 Why Patch Size Matters

Consider the same:

$$
224\times224
$$

image.

### 16×16 patches

$$
14\times14=196
$$

tokens.

### 32×32 patches

$$
7\times7=49
$$

tokens.

Therefore:

```text
Smaller patch
      ↓
More tokens
      ↓
Finer spatial information
      ↓
Higher sequence length
```

while:

```text
Larger patch
      ↓
Fewer tokens
      ↓
Coarser spatial information
      ↓
Lower sequence length
```

This matters greatly because Transformer self-attention becomes more expensive as sequence length increases. We will study that computational relationship in detail later.

---

# 13. 💡 Why Not Make Every Pixel a Token?

For a 224×224 image:

$$
224\times224=50,176
$$

spatial positions.

If every pixel became a token:

$$
\approx50,176
$$

tokens.

With 16×16 patches:

$$
\boxed{196}
$$

patch tokens.

So patchification changes:

```text
50,176 pixel positions
          ↓
       196 patches
```

That is an enormous reduction in sequence length.

This is one important reason patchification makes Transformer-based image processing practical. Research on patch-based vision models also highlights the relationship between patch representations and the quadratic cost of self-attention. ([arXiv][4])

---

# 14. 🏗️ Complete ViT Input Pipeline

```text
                 IMAGE
            224 × 224 × 3
                    │
                    ▼
            ┌──────────────┐
            │ Patchify     │
            │ 16 × 16      │
            └──────────────┘
                    │
                    ▼
               196 patches
                    │
                    ▼
          Linear Patch Projection
                    │
                    ▼
             196 × 768
                    │
                    ▼
               Add [CLS]
                    │
                    ▼
             197 × 768
                    │
                    ▼
        Add Positional Embeddings
                    │
                    ▼
          Transformer Encoder
                    │
                    ▼
              [CLS] vector
                    │
                    ▼
          Classification Head
                    │
                    ▼
                Prediction
```

---

# 15. 🖼️ Architecture Visual

![Image](https://images.openai.com/static-rsc-4/NwEbGnjdvtme4n878q1HrUEU7BLkb7CFcbXdwc36b8qam7WfKtmZTmt4V4K1svKIJeVS_EdBNiRj4E-7sog920v481rHDHzXghodYFQXaMj-Mm5-VRw9iOk_ol2yyeXMJyaVqqc2UaMrbNgdNHDRoefAScuJ5dbmimVSWkKuytrtcFWymetnfULxloulF3eF?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/kbcrV8FloklnE_Vq-fJ6NP3lKAmXRBL86ugwmXFTEDJJXPPWG6SL_A5_y89hPLFbruyH6X9K7aV0HBfID8Obtu8RT8K8ARl0AYx0rXkkmiVaJlWorebjbg1e6yJFk55CYcIOwX03IFrwNeOdC49kfqYof1tX-78WnywHC-vq3ec1VS0Jz6trzihIiSCUdT3I?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/Dh6hpnpHTfWqYQcnGvl5kcDWG_HadQtySoVF70bGm8yf1Qw3XHHZwl7U-quIHRYIla0jJ_h7SjO0vvNjLcqjPLmSz8V1G5EFIxfzfR9ixGNri35oMraYMNFSuwgFAn-Q0_8V3MGDEk4ehE52HSFrpbSYOM-JJcZ2XuRNpiaSIfbtAlQgUmtjVcCj_9TeYXzc?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/5kROLarmaPQIZDLIAhXwAzkrKmwsOpnNrIQRtfZzSfkxxO1uGkvycVdgimu4LWBgcpXBGdIiogFNg3kuWnD9D1IZVU8vWA8m-sykrUCWhmbHSrwacKk94gDiLYZkmEc0edJBzsOjcW3nhDB9ik2hwhQPhfnUSqz9jVb2va1uztNSXECHhdmENM0JwvAZB25p?purpose=fullsize)

The original ViT paper's architecture figure is the key visual to remember for this section. ([arXiv][1])

---

# 16. 🧪 PyTorch Mental Model

Later, when implementing ViT, the patchification stage can conceptually be represented as:

```text
Input
[B, 3, 224, 224]
        ↓
Patch Embedding
        ↓
[B, 196, 768]
        ↓
CLS token
        ↓
[B, 197, 768]
        ↓
Position embedding
        ↓
[B, 197, 768]
        ↓
Transformer
```

Modern implementations may perform patch projection using a convolution with:

```text
kernel_size = patch_size
stride      = patch_size
```

This produces non-overlapping patch embeddings directly. Hugging Face's implementation, for example, uses a `Conv2d` projection with kernel size and stride equal to the patch size. ([Hugging Face][5])

⚠️ This implementation detail is useful to know, but **do not confuse the projection layer with a CNN backbone**. Its role here is to create patch embeddings.

---

# 17. 🎯 Three Concepts You MUST Know

## ① Patchification

```text
Image
 ↓
Non-overlapping patches
```

## ② Patch Embedding

```text
Patch
 ↓
Learned vector representation
```

## ③ Positional Embedding

```text
Patch embedding
 +
Location information
 ↓
Transformer input
```

Together:

```text
IMAGE
 ↓
PATCHES
 ↓
PATCH EMBEDDINGS
 ↓
+ [CLS]
 ↓
+ POSITION
 ↓
TRANSFORMER
```

---

# 🎤 Interview Questions

### Q1. Why does ViT divide images into patches?

To convert the 2D image into a sequence of visual tokens that can be processed by a Transformer.

### Q2. How many 16×16 patches are in a 224×224 image?

$$
\boxed{196}
$$

### Q3. How many raw values are in one 16×16 RGB patch?

$$
16\times16\times3=\boxed{768}
$$

### Q4. What does patch embedding do?

It transforms a raw image patch into a learned vector representation suitable for the Transformer.

### Q5. Why are positional embeddings needed?

Because the Transformer needs information about the spatial position of each patch.

### Q6. What is `[CLS]`?

A learnable special token whose final representation is used as a whole-image representation for classification in the original ViT design.

### Q7. What happens when patch size decreases?

The number of tokens increases, providing finer spatial granularity but increasing computational cost.

---

# 🚨 Common Mistakes

### ❌ Mistake 1

> "224/16 = 14 patches."

No.

That's **14 patches per dimension**.

$$
14\times14=\boxed{196}
$$

---

### ❌ Mistake 2

> "The patch itself is the embedding."

Not exactly.

```text
Raw patch
   ↓
Projection
   ↓
Patch embedding
```

---

### ❌ Mistake 3

> "Patch embedding tells the model where the patch is."

No.

* Patch embedding → **visual content**
* Positional embedding → **spatial location**

---

### ❌ Mistake 4

> "`[CLS]` is one of the image patches."

No.

It is a **separate learnable token**.

---

### ❌ Mistake 5

> "ViT treats every pixel as a token."

The original ViT uses **patch tokens**, not individual pixel tokens.

---

### ❌ Mistake 6

> "Patch size only affects memory."

No. It affects:

* number of tokens
* spatial granularity
* sequence length
* attention computation

---

# 🧠 30.2 — One-Minute Revision

```text
224 × 224 × 3 IMAGE
          │
          ▼
     16 × 16 PATCHES
          │
          ▼
       196 PATCHES
          │
          ▼
    PATCH EMBEDDINGS
          │
          ▼
       + [CLS]
          │
          ▼
  POSITIONAL EMBEDDINGS
          │
          ▼
   197 × 768 SEQUENCE
          │
          ▼
 TRANSFORMER ENCODER
          │
          ▼
    [CLS] REPRESENTATION
          │
          ▼
 CLASSIFICATION HEAD
```

### 🔥 Formula

$$
\boxed{N=\frac{HW}{P^2}}
$$

### 🔥 Remember

> **Patch embedding = WHAT**

> **Positional embedding = WHERE**

> **`[CLS]` = WHOLE IMAGE REPRESENTATION**

> **Transformer = LEARN RELATIONSHIPS**

---

# 🚀 QUICK REVISION CHEAT SHEET

| Concept              | Key point                                             |
| -------------------- | ----------------------------------------------------- |
| Patch                | Fixed-size image region                               |
| Patchification       | Image → sequence of patches                           |
| Patch count          | \(N=HW/P^2\)                                          |
| 224×224 + 16×16      | 196 patches                                           |
| RGB 16×16 patch      | 768 raw values                                        |
| Patch embedding      | Raw patch → learned vector                            |
| `[CLS]`              | Whole-image representation for classification         |
| Positional embedding | Provides patch location                               |
| ViT-Base example     | 768-dimensional embeddings                            |
| 16×16 patches        | 196 patch tokens                                      |
| With `[CLS]`         | 197 tokens                                            |
| Smaller patches      | More tokens + finer spatial granularity               |
| Larger patches       | Fewer tokens + coarser spatial granularity            |
| Main pipeline        | Image → patches → embeddings → position → Transformer |

### 🧠 Final mental model

> **ViT takes an image, cuts it into patches, converts each patch into a vector, adds information about where each patch came from, and feeds the resulting visual-token sequence into a Transformer.**

**Primary reference:** the original ViT paper, *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*. ([arXiv][1])

---